# OneClickAI 이미지 분류
이 노트북은 Google Colab 환경에서 실행하도록 작성되었습니다.

**순서**
1. 파일 업로드 및 데이터 확인
2. 데이터 전처리 및 모델 학습
3. 결과 확인
4. 웹캠으로 예측

## 1. 파일 업로드 및 데이터 확인

클래스별 이미지가 폴더로 분류된 zip 파일을 업로드해 주세요.

```
dataset.zip
└── dataset/
    ├── 클래스1/
    │   └── image1.jpg
    └── 클래스2/
        └── image1.jpg
```

In [ ]:
# ==========================================
# 1. 파일 업로드 및 압축 풀기 (Colab 환경)
# ==========================================
from google.colab import files
import zipfile
import os
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
import matplotlib.pyplot as plt
import numpy as np
import cv2

print("이미지 데이터가 담긴 압축 파일(.zip)을 업로드 해주세요.")
uploaded = files.upload()

extract_dir = "dataset"
for file_name in uploaded.keys():
    with zipfile.ZipFile(file_name, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

# 폴더 목록에서 클래스 이름을 가져옴
class_names = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]

num_classes = len(class_names)
print(f'탐색된 클래스: {class_names}')


# ==========================================
# 데이터 불러오기 및 확인
# ==========================================
# 각 클래스별로 이미지 개수 확인
for name in class_names:
    path = os.path.join(extract_dir, name)
    count = len([f for f in os.listdir(path) if f.endswith(('.jpg', '.png'))])
    print(f'{name} 이미지 개수: {count}')


## 2. 데이터 전처리 및 모델 학습

이미지를 96×96으로 리사이즈하고 정규화한 뒤 CNN 모델을 학습합니다.

In [ ]:

# ==========================================
# 데이터 전처리 (이미지 정규화, 사이즈 변경)
# ==========================================
x = []
y = []

# 모든 클래스에 대해 이미지 로드 및 전처리
for idx, name in enumerate(class_names):
    class_path = os.path.join(extract_dir, name)
    imgs = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png'))]
    for img_name in imgs:
        img = cv2.imread(os.path.join(class_path, img_name))
        if img is None: continue

        img = cv2.resize(img, (96, 96)) # 이미지 사이즈 변경 (96,96)
        x.append(img/127.5-1)           # 이미지 정규화 (-1 ~ 1 사이로)
        y.append(idx)                   # 해당 클래스의 인덱스를 라벨로 사용

# 리스트 -> Array로 변경
x = np.stack(x)
y = np.array(y)

# 데이터 섞기 (예를 들어 강아지, 고양이가 순서대로 뭉쳐있으므로 학습 전 섞어주기)
indices = np.arange(x.shape[0])
np.random.shuffle(indices)
x = x[indices]
y = y[indices]

# ==========================================
# 데이터를 training set과 validation set으로 나누기
# ==========================================
# 업로드한 데이터 개수(10개)에 맞춰 80%는 학습, 20%는 검증으로 나눕니다.
split_idx = int(len(x) * 0.8)

x_train = x[:split_idx, :, :, :]
y_train = y[:split_idx]
x_val = x[split_idx:, :, :, :]
y_val = y[split_idx:]

# onehot으로 변경
# 예를 들면 강아지의 경우 (0,1), 고양이의 경우 (1,0)
y_train = tf.one_hot(y_train, num_classes)
y_val = tf.one_hot(y_val, num_classes)

# ==========================================
# 모델 생성
# ==========================================
model = tf.keras.Sequential()
model.add(Conv2D(16, (3, 3), input_shape=(96, 96, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(16, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

# ==========================================
# 모델 설정 및 학습
# ==========================================
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), batch_size=8, epochs=50)

# ==========================================
# 모델 저장 및 불러오기
# ==========================================
model.save('image_classifier_model.keras')
model_loaded = tf.keras.models.load_model('image_classifier_model.keras')


## 3. 결과 확인

학습 과정의 손실(Loss) 그래프를 출력하고, 학습 데이터로 예측 결과를 확인합니다.

In [ ]:
# ==========================================
# 모델 결과 확인 (그래프)
# ==========================================
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper right')
plt.show()

# ==========================================
# 테스트 및 시각화
# ==========================================
# 첫 번째 학습 데이터로 예측 테스트
test_idx = 0
pred = model.predict(np.array([x_train[test_idx]]))
print("예측 결과 (확률):", pred)

# 정규화된 이미지를 원래 색상(0~255)으로 되돌려서 화면에 출력
test_img = ((x_train[test_idx] + 1) * 127.5).astype(np.uint8)
plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
plt.title('Test Image')
plt.show()



## 4. 웹캠으로 예측

웹캠으로 사진을 찍어 학습된 모델로 분류합니다.  
팝업 창에서 **Capture** 버튼을 클릭해 주세요.

In [ ]:
# ==========================================
# 웹캠 연결 및 예측 추가
# ==========================================
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

print("웹캠을 사용하여 이미지를 캡처하고 모델로 예측합니다.")
print("팝업되는 창에서 'Capture' 버튼을 클릭하여 이미지를 찍어주세요.")

try:
  filename = take_photo()
  print('Saved to {}'.format(filename))

  # Read the image and preprocess for the model
  img = cv2.imread(filename)
  if img is None:
    print("Error: Could not read image from webcam.")
  else:
    # Resize the image to 96x96
    img_resized = cv2.resize(img, (96, 96))
    # Normalize the image (-1 to 1)
    img_normalized = (img_resized / 127.5) - 1
    # Add batch dimension
    img_input = np.expand_dims(img_normalized, axis=0)

    # Make a prediction using the loaded model
    pred = model_loaded.predict(img_input)
    print("Webcam image prediction (probabilities):", pred)

    # Get the predicted class name
    predicted_class_idx = np.argmax(pred)
    predicted_class_name = class_names[predicted_class_idx]
    print(f"Predicted class: {predicted_class_name}")

    # Display the image for verification
    plt.imshow(cv2.cvtColor(((img_normalized + 1) * 127.5).astype(np.uint8), cv2.COLOR_BGR2RGB))
    plt.title(f'Webcam Captured Image - Predicted: {predicted_class_name}')
    plt.show()

except Exception as err:
  print(f"An error occurred: {str(err)}")
  print("Please ensure you have a webcam and grant permission when prompted.")
